# Day 03：流程控制（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 1 周**：第 1 周：Python 核心语法与数据结构基石  
> **GitLab 工单**：`STARMIND-003` ｜ **开发分支**：`feat/day03-control-flow-cli`  
> **配套 HTML5 课件**：`courseware/day03/index.html` ｜ **独立版本快照**：`versions/day03/`

---

## 🎙️ 老陈开场旁白

前两天咱们写的都是「一条道走到黑」的顺序代码，今天要让程序长出脑子——学会做判断和干重复活！林产品刚提了 STARMIND-003：客服工单进来后，要根据紧急程度和金额自动分流给「人工专家」「高级模型」或「轻量模型」；另外为了方便运营同学在终端里试用工具，需要做一个命令行交互菜单，顺便把经典的「猜数字推理测试」和「九九配额表」做成演示命令。重点提醒一下：写循环控制台时，千万要把交互逻辑封装成函数，支持传入模拟指令列表跑自动化测试，不然阿杰跑 CI 时直接卡死在 `input()` 里！

### 承上启下
- **昨日基线**：昨天咱们完成了工单文本清洗器与 f-string Prompt 模板引擎，洗干净了单条输入数据。
- **明日预告**：明天（Day 4）咱们升级数据容器，学习列表（list）、元组（tuple）和集合（set），给星辰智脑做出待办事项管理器与标签去重引擎。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day03'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-003）

星辰智脑每天面对高价值客诉、普通咨询、垃圾广告三类混合工单，全部调用昂贵的旗舰大模型会导致算力成本失控，需要基于规则的流程路由与终端运营控制台。

**今日迭代目标：**
- 精通 `if / elif / else` 多分支路由与卫语句降嵌套技巧。
- 掌握 `for`、`while`、`range`、`break`、`continue` 在批处理与交互菜单中的应用。
- 完成工单路由分发器、九九算力表、猜数字二分验证器与可测试命令行菜单。

**验收标准：**
- 能按金额、关键词与客户等级将工单准确路由至 L1/L2/L3 通道。
- 支持批量工单扫描：遇空工单 `continue` 跳过，遇高危攻击 `break` 触发熔断。
- CLI 菜单支持传入模拟命令序列自动执行并正常退出，pytest 全部通过。

---

## 📚 课堂核心技术笔记

### 1. 1. if/elif/else 与卫语句降嵌套

多分支判断按「最高优先级/最特殊条件」写在最前面。当某个异常条件不满足时，直接 `return` 返回结果（卫语句），能把 4 层缩进压平到 1 层，代码可读性瞬间提升。

> **⚠️ 老陈避坑**：注意 `if` 和 `elif` 是互斥的，只会命中第一个为 True 的分支；如果误把多个 `elif` 全写成 `if`，会导致重复触发多个路由！

### 2. 2. for / range 与 while 循环选型

已知遍历集合或确定次数时优先用 `for ... in ...` 和 `range(start, stop, step)`（同样左闭右开）；未知迭代次数、需要等待特定状态结束时（如交互菜单、重试机制）使用 `while`。

> **⚠️ 老陈避坑**：使用 `while` 循环时，务必确保循环体内有改变终止条件的语句，或在生产代码中加上 `max_steps` 安全护栏，防止线上 CPU 跑满 100%。

### 3. 3. break、continue 与循环的 else 子句

`continue` 跳过当前这一次迭代进入下一轮；`break` 直接打断当前所在的最内层循环。Python 特有的 `for ... else:` 语法：只有当循环「正常跑完、没有被 `break` 打断」时，才会执行 `else` 块，非常适合做批量校验后的「全部安全通过」确认。

> **⚠️ 老陈避坑**：嵌套循环里，`break` 只会跳出离它最近的那一层循环，不会直接跳出外层循环。


## 💻 实操步骤 1：步骤一：实现工单三级分流路由器与批处理熔断引擎

- **目标文件**：`versions/day03/src/day03_ticket_router.py`
- **老陈代码旁白**：先搞定林产品的工单分流需求。咱们用卫语句写单条路由，再用 `for + continue + break + else` 写批量扫描熔断逻辑。


In [ ]:
"""
STARMIND-003 模块一：工单三级智能路由与批处理熔断引擎。
演示 if/elif/else、卫语句、for 循环、continue 跳过、break 熔断与 for-else 语法。
"""

from typing import List, Dict, Any


def route_single_ticket(text: str, amount: float, is_vip: bool) -> str:
    """
    根据工单内容、涉及金额与客户等级，决定分发通道：
    - L3_HUMAN: 投诉/法律风险 或 涉案金额 >= 5000 元
    - L2_FLAGSHIP_LLM: VIP 客户 或 金额 >= 500 元
    - L1_FAST_LLM: 普通日常咨询
    """
    cleaned = text.strip()
    # 卫语句 1：空工单直接返回无效标记
    if not cleaned:
        return "INVALID"

    # 卫语句 2：高风险关键词或大额纠纷直接转人工专家
    if "投诉" in cleaned or "起诉" in cleaned or amount >= 5000:
        return "L3_HUMAN"

    # VIP 客户或中等金额由旗舰大模型处理
    if is_vip or amount >= 500:
        return "L2_FLAGSHIP_LLM"

    # 其余走低成本轻量大模型
    return "L1_FAST_LLM"


def process_ticket_batch(tickets: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    批量扫描工单队列：
    - 空文本工单：continue 跳过并计入 skipped
    - 包含 'ATTACK_PAYLOAD' 的恶意攻击单：立即 break 触发全队列熔断
    """
    stats: Dict[str, Any] = {
        "L1_FAST_LLM": 0,
        "L2_FLAGSHIP_LLM": 0,
        "L3_HUMAN": 0,
        "skipped": 0,
        "circuit_broken": False,
    }

    for ticket in tickets:
        text = str(ticket.get("text", "")).strip()
        # 空工单使用 continue 跳过，不消耗路由资源
        if not text:
            stats["skipped"] += 1
            continue

        # 检测到批量注入攻击特征，立即 break 熔断保护下游
        if "ATTACK_PAYLOAD" in text:
            stats["circuit_broken"] = True
            break

        channel = route_single_ticket(
            text=text,
            amount=float(ticket.get("amount", 0.0)),
            is_vip=bool(ticket.get("is_vip", False)),
        )
        stats[channel] += 1
    else:
        # 仅当循环未被 break 打断时执行，确认批次完整扫描
        stats["completed_cleanly"] = True

    stats.setdefault("completed_cleanly", False)
    return stats


if __name__ == "__main__":
    sample_batch = [
        {"text": "请问怎么修改发票抬头？", "amount": 0, "is_vip": False},
        {"text": "   ", "amount": 0, "is_vip": False},
        {"text": "订单延迟发货我要投诉！", "amount": 200, "is_vip": False},
        {"text": "VIP 专属套餐续费咨询", "amount": 1200, "is_vip": True},
    ]
    print("批次处理统计:", process_ticket_batch(sample_batch))


### 🔍 步骤 1 代码深度解析

1. `route_single_ticket` 全程用卫语句提前 `return`，没有一层冗余的 `else` 嵌套。
2. `for ... else:` 结构在这里非常地道：正常跑完自动打上 `completed_cleanly = True`；一旦被 `break` 熔断则跳过 `else`。

**预期运行输出参考：**
```text
批次处理统计: {'L1_FAST_LLM': 1, 'L2_FLAGSHIP_LLM': 1, 'L3_HUMAN': 1, 'skipped': 1, 'circuit_broken': False, 'completed_cleanly': True}
```


## 💻 实操步骤 2：步骤二：实现猜数字沙盒、九九乘法表与可测试 CLI 控制台

- **目标文件**：`versions/day03/src/day03_cli_menu.py`
- **老陈代码旁白**：现在把猜数字、九九乘法表和星辰智脑控制台整合进一个 `while` 菜单里。注意看我怎么通过传入指令序列，让它既能手动玩又能秒跑自动化测试。


In [ ]:
"""
STARMIND-003 模块二：命令行交互菜单、九九乘法表与猜数字验证沙盒。
演示嵌套 for 循环、while 交互循环以及可测试的 CLI 命令分发设计。
"""

from typing import List, Tuple


def build_multiplication_table(n: int = 9) -> str:
    """使用嵌套 for 与 range 生成标准对齐的九九乘法表字符串。"""
    lines: List[str] = []
    # 外层控制行号 1..n
    for row in range(1, n + 1):
        row_items: List[str] = []
        # 内层控制列号 1..row，形成下三角矩阵
        for col in range(1, row + 1):
            row_items.append(f"{col}x{row}={col * row:<2d}")
        lines.append("  ".join(row_items))
    return "\n".join(lines)


def play_guess_number(secret: int, guesses: List[int], max_attempts: int = 5) -> Tuple[bool, List[str]]:
    """
    猜数字核心引擎（非阻塞可测试版）。
    返回 (是否猜中, 每轮反馈日志列表)。
    """
    logs: List[str] = []
    attempts = 0

    for guess in guesses:
        attempts += 1
        if attempts > max_attempts:
            logs.append(f"已超过最大尝试次数 ({max_attempts})，游戏结束！")
            break

        if guess < secret:
            logs.append(f"第 {attempts} 次猜 {guess}：偏小了，往大猜")
        elif guess > secret:
            logs.append(f"第 {attempts} 次猜 {guess}：偏大了，往小猜")
        else:
            logs.append(f"第 {attempts} 次猜 {guess}：恭喜命中目标值 {secret}！")
            return True, logs

    return False, logs


def run_cli_menu(commands: List[str]) -> List[str]:
    """
    星辰智脑命令行交互菜单调度器。
    传入命令列表模拟用户输入：
      '1' -> 查看九九乘法表（前3行预览）
      '2' -> 运行内置猜数字演示
      'q' -> 退出控制台
    """
    outputs: List[str] = []
    idx = 0

    # 使用 while 循环逐条消费指令，带边界索引防止死循环
    while idx < len(commands):
        cmd = commands[idx].strip().lower()
        idx += 1

        if cmd == "1":
            table = build_multiplication_table(3)
            outputs.append(f"[乘法表预览]\n{table}")
        elif cmd == "2":
            won, logs = play_guess_number(secret=42, guesses=[30, 50, 42])
            outputs.append(f"[猜数字演示] 结果={'成功' if won else '失败'} | 步骤数={len(logs)}")
        elif cmd in ("q", "exit", "quit"):
            outputs.append("已安全退出星辰智脑控制台，再见！")
            break
        else:
            outputs.append(f"未知指令 '{cmd}'，请输入 1 / 2 / q")

    return outputs


if __name__ == "__main__":
    print("=== 完整九九乘法表 ===")
    print(build_multiplication_table(9))
    print("\n=== 模拟 CLI 菜单会话 ===")
    session_logs = run_cli_menu(["1", "2", "invalid", "q"])
    for entry in session_logs:
        print(entry)


### 🔍 步骤 2 代码深度解析

1. `f'{col}x{row}={col * row:<2d}'` 中的 `:<2d` 让个位数自动左对齐补空格，打印出来的九九乘法表整整齐齐。
2. `run_cli_menu(commands)` 把交互驱动和业务分支彻底分开，既能演示 `while` + `break` 的菜单控制流，又能在测试套件里零卡顿跑完所有分支。

**预期运行输出参考：**
```text
=== 完整九九乘法表 ===
1x1=1 
1x2=2   2x2=4 
...
=== 模拟 CLI 菜单会话 ===
[乘法表预览]
1x1=1 
1x2=2   2x2=4 
1x3=3   2x3=6   3x3=9 
[猜数字演示] 结果=成功 | 步骤数=3
未知指令 'invalid'，请输入 1 / 2 / q
已安全退出星辰智脑控制台，再见！
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day03/tests/test_day03.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-003 自动化单元测试。"""

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day03_ticket_router import route_single_ticket, process_ticket_batch
from day03_cli_menu import build_multiplication_table, play_guess_number, run_cli_menu


def test_route_single_ticket():
    assert route_single_ticket("   ", 0, False) == "INVALID"
    assert route_single_ticket("我要投诉服务差", 10, False) == "L3_HUMAN"
    assert route_single_ticket("大额订单异常", 6000, False) == "L3_HUMAN"
    assert route_single_ticket("普通咨询", 100, True) == "L2_FLAGSHIP_LLM"
    assert route_single_ticket("查物流", 50, False) == "L1_FAST_LLM"


def test_batch_circuit_breaker():
    batch = [
        {"text": "正常问题", "amount": 10, "is_vip": False},
        {"text": "", "amount": 0, "is_vip": False},
        {"text": "ATTACK_PAYLOAD_999", "amount": 0, "is_vip": False},
        {"text": "熔断后不应被处理", "amount": 9999, "is_vip": True},
    ]
    res = process_ticket_batch(batch)
    assert res["L1_FAST_LLM"] == 1
    assert res["skipped"] == 1
    assert res["circuit_broken"] is True
    assert res["completed_cleanly"] is False
    assert res["L3_HUMAN"] == 0


def test_multiplication_and_guess_and_cli():
    table = build_multiplication_table(9)
    assert "9x9=81" in table
    assert len(table.splitlines()) == 9

    won, logs = play_guess_number(secret=50, guesses=[25, 75, 50])
    assert won is True
    assert len(logs) == 3

    lost, _ = play_guess_number(secret=50, guesses=[1, 2, 3], max_attempts=2)
    assert lost is False

    out = run_cli_menu(["1", "2", "unknown", "q", "1"])
    assert len(out) == 4  # q 之后的 '1' 不应执行
    assert "安全退出" in out[-1]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：API 限流退避重试模拟器

**题目要求：**
大模型接口偶发 429 限流，需要最多重试 4 次。编写 `simulate_retry(statuses: list[int]) -> tuple[bool, int]`，遍历状态码列表：遇到 200 立刻返回 `(True, 尝试次数)`；遇到 429 则继续重试；遇到 401（鉴权失败）立即 `break` 放弃重试并返回 `(False, 尝试次数)`。

**思路提示：** 用 `enumerate(statuses, start=1)` 追踪次数，区分可重试错误与不可重试错误。


In [ ]:
"""Day 3 课后题 1：API 限流重试策略模拟器。"""

from typing import List, Tuple


def simulate_retry(statuses: List[int], max_retries: int = 4) -> Tuple[bool, int]:
    """模拟接口重试：200 成功，429 继续，401 鉴权失败直接放弃。"""
    attempts = 0
    for code in statuses[:max_retries]:
        attempts += 1
        if code == 200:
            return True, attempts
        if code == 401:
            # 密钥错误重试多少次都没用，直接 break 止损
            break
    return False, attempts


if __name__ == "__main__":
    print("场景1(重试成功):", simulate_retry([429, 429, 200]))
    print("场景2(鉴权熔断):", simulate_retry([429, 401, 200]))


**答案解析与评分标准：**

提前演练真实企业 API 调用中的「可重试 vs 不可重试错误」分类思想，避免对 401/400 错误盲目重试浪费配额。


### 作业 2（企业实战）：企业实战：Token 预算水位监控器

**题目要求：**
给定一组每日消耗 Token 列表，累计总消耗并在超过月度预算 `budget` 时停止扣费，返回已成功处理的请求数和剩余预算。

**思路提示：** 循环中先判断 `spent + cost > budget`，若超限则 `break`，否则累加。


In [ ]:
"""Day 3 课后题 2：月度 Token 预算熔断监控器。"""

from typing import List, Dict


def consume_budget(costs: List[int], budget: int) -> Dict[str, int]:
    """按顺序扣减 Token 预算，即将超限时停止执行。"""
    spent = 0
    processed = 0
    for cost in costs:
        if cost < 0:
            continue
        if spent + cost > budget:
            break
        spent += cost
        processed += 1
    return {"processed": processed, "spent": spent, "remaining": budget - spent}


if __name__ == "__main__":
    print(consume_budget([3000, 4000, 5000, 2000], budget=10000))


**答案解析与评分标准：**

考查循环内前瞻检查（先判超限再扣减）与负值脏数据跳过。


### 作业 3（架构进阶）：架构进阶：二分查找自动猜数字求解器

**题目要求：**
编写一个自动求解函数 `binary_search_guess(secret: int, low: int = 1, high: int = 100) -> list[int]`，用二分法在最多 7 步内猜中 `[1, 100]` 范围内的任意目标整数，并返回猜测路径。

**思路提示：** 每次取 `mid = (low + high) // 2`，偏小则 `low = mid + 1`，偏大则 `high = mid - 1`。


In [ ]:
"""Day 3 课后题 3：二分查找自动猜数求解器。"""

from typing import List


def binary_search_guess(secret: int, low: int = 1, high: int = 100) -> List[int]:
    """使用二分法定位目标值，返回完整猜测序列。"""
    path: List[int] = []
    while low <= high:
        mid = (low + high) // 2
        path.append(mid)
        if mid == secret:
            break
        elif mid < secret:
            low = mid + 1
        else:
            high = mid - 1
    return path


if __name__ == "__main__":
    trace = binary_search_guess(73)
    print(f"猜测路径: {trace}，仅用 {len(trace)} 步命中！")


**答案解析与评分标准：**

用 `while` 循环实现 O(log N) 二分搜索，100 个数最多 7 步，100 万个数最多 20 步，体会算法思维对循环效率的巨大提升。


---

## 🎯 老陈收尾复盘

今天你已经掌握了编程世界最核心的三大结构：顺序、分支、循环。记住老陈的「卫语句（Guard Clause）」原则：条件判断尽量提前 return，别写出五六层缩进的「箭头代码」，看着头疼还容易漏分支。晚上把菜单和分流逻辑再过一遍，明天咱们进入核心数据结构！
